# 21. LSTM으로 PM10 다음 시간 예측하기

이번 장에서는 과거 24시간 PM10 값을 보고 다음 1시간의 PM10 값을 예측합니다.

이번 문제는 회귀입니다.

```text
입력 X = 과거 24시간 PM10
정답 y = 다음 시간 PM10
```

## 1. 라이브러리 불러오기

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error

from keras.models import Sequential
from keras.layers import Input, LSTM, Dense, Dropout
from keras.callbacks import EarlyStopping

## 2. PM10 데이터 준비

In [ ]:
DATA_PATH = Path(r"C:\work\dataset\seoul_pm10.csv")
TARGET_AREA = "강남구"

df = pd.read_csv(DATA_PATH, encoding="cp949")
df["date"] = pd.to_datetime(df["date"], errors="coerce")

area_df = df[df["area"] == TARGET_AREA].copy()
area_df = area_df.sort_values("date")
area_df["pm10_filled"] = area_df["pm10"].ffill()
area_df = area_df.dropna(subset=["pm10_filled"])

print("선택 지역:", TARGET_AREA)
print("정리 후 shape:", area_df.shape)
area_df.head()

## 3. 시간 순서대로 train/validation 나누기

In [ ]:
# PM10 값을 2차원 배열로 만듭니다.
# MinMaxScaler는 2차원 입력을 기대합니다.
pm10_values = area_df["pm10_filled"].values.reshape(-1, 1)

split_index = int(len(pm10_values) * 0.8)

train_values = pm10_values[:split_index]
val_values = pm10_values[split_index:]

print("train 길이:", len(train_values))
print("validation 길이:", len(val_values))

## 4. 스케일링

In [ ]:
scaler = MinMaxScaler()

# train 데이터로만 최솟값/최댓값을 학습합니다.
train_scaled = scaler.fit_transform(train_values)

# validation은 train에서 배운 기준으로 변환만 합니다.
val_scaled = scaler.transform(val_values)

print("train_scaled 범위:", train_scaled.min(), "~", train_scaled.max())
print("val_scaled 범위:", val_scaled.min(), "~", val_scaled.max())

## 5. window 데이터셋 만들기

In [ ]:
def make_window_dataset(values, window_size):
    """과거 window_size개 값을 입력 X로, 그 다음 값을 정답 y로 만듭니다."""
    X = []
    y = []
    
    for i in range(len(values) - window_size):
        X.append(values[i:i + window_size])
        y.append(values[i + window_size])
    
    return np.array(X), np.array(y)


WINDOW_SIZE = 24

X_train, y_train = make_window_dataset(train_scaled, WINDOW_SIZE)
X_val, y_val = make_window_dataset(val_scaled, WINDOW_SIZE)

print("X_train shape:", X_train.shape)
print("y_train shape:", y_train.shape)
print("X_val shape:", X_val.shape)
print("y_val shape:", y_val.shape)

## 6. LSTM 모델 만들기

입력 모양은 `(24, 1)`입니다.

```text
24 = 과거 24시간
1 = PM10 특성 하나
```

In [ ]:
model = Sequential([
    # 각 샘플은 과거 24시간, 각 시간에는 PM10 값 1개를 가집니다.
    Input(shape=(WINDOW_SIZE, 1)),
    
    # LSTM은 시간 순서가 있는 입력을 읽는 순환 신경망 계열 층입니다.
    LSTM(32),
    
    # 과적합을 줄이기 위해 일부 출력을 학습 중 임시로 끕니다.
    Dropout(0.2),
    
    # 회귀 출력층입니다.
    # 다음 시간 PM10 값 하나를 예측하므로 Dense(1)을 사용합니다.
    Dense(1)
])

model.summary()

## 7. 모델 컴파일

PM10 예측은 숫자를 맞히는 회귀 문제입니다.

그래서 손실 함수로 `mse`를 사용합니다.

In [ ]:
model.compile(
    optimizer="adam",
    loss="mse",
    metrics=["mae"]
)

## 8. EarlyStopping 설정

In [ ]:
early_stopping = EarlyStopping(
    monitor="val_loss",
    patience=3,
    restore_best_weights=True
)

## 9. 모델 학습

CPU 환경을 고려해 epoch는 작게 시작합니다.

In [ ]:
history = model.fit(
    X_train,
    y_train,
    validation_data=(X_val, y_val),
    epochs=20,
    batch_size=32,
    callbacks=[early_stopping],
    verbose=1
)

## 10. 학습 곡선 확인

In [ ]:
plt.figure(figsize=(10, 4))

plt.subplot(1, 2, 1)
plt.plot(history.history["loss"], label="train loss")
plt.plot(history.history["val_loss"], label="validation loss")
plt.title("MSE Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history.history["mae"], label="train mae")
plt.plot(history.history["val_mae"], label="validation mae")
plt.title("MAE")
plt.xlabel("Epoch")
plt.ylabel("MAE")
plt.legend()

plt.tight_layout()
plt.show()

## 11. 검증 데이터 예측

In [ ]:
# predict()는 검증 입력에 대한 예측값을 만듭니다.
y_val_pred_scaled = model.predict(X_val, verbose=0)

print("예측값 shape:", y_val_pred_scaled.shape)
print("앞 5개 예측값:")
print(y_val_pred_scaled[:5].ravel())

## 12. 원래 PM10 단위로 되돌리기

모델 예측값은 0~1 범위로 스케일링된 값입니다.

사람이 해석하려면 원래 PM10 단위로 되돌려야 합니다.

In [ ]:
# inverse_transform()은 MinMaxScaler로 바꿨던 값을 원래 단위로 되돌립니다.
y_val_original = scaler.inverse_transform(y_val)
y_val_pred_original = scaler.inverse_transform(y_val_pred_scaled)

print("실제 PM10 앞 5개:")
print(y_val_original[:5].ravel())

print("\n예측 PM10 앞 5개:")
print(y_val_pred_original[:5].ravel())

## 13. MAE와 RMSE 계산

In [ ]:
mae = mean_absolute_error(y_val_original, y_val_pred_original)

# mean_squared_error의 제곱근을 구하면 RMSE입니다.
rmse = np.sqrt(mean_squared_error(y_val_original, y_val_pred_original))

print(f"검증 MAE: {mae:.2f}")
print(f"검증 RMSE: {rmse:.2f}")

## 14. 실제값과 예측값 비교 그래프

처음 200개 검증 시점만 그려서 비교합니다.

In [ ]:
PLOT_COUNT = 200

plt.figure(figsize=(12, 4))
plt.plot(y_val_original[:PLOT_COUNT], label="actual PM10")
plt.plot(y_val_pred_original[:PLOT_COUNT], label="predicted PM10")
plt.title("Actual vs Predicted PM10")
plt.xlabel("Validation time step")
plt.ylabel("PM10")
plt.legend()
plt.tight_layout()
plt.show()

## 15. 예측 오차 보기

In [ ]:
errors = y_val_pred_original.ravel() - y_val_original.ravel()

plt.figure(figsize=(8, 4))
plt.hist(errors, bins=30)
plt.title("Prediction Error Distribution")
plt.xlabel("Prediction - Actual")
plt.ylabel("Count")
plt.tight_layout()
plt.show()

print("오차 평균:", errors.mean())
print("오차 표준편차:", errors.std())

## 16. 모델 저장

In [ ]:
MODEL_DIR = Path(r"C:\work\deepLearning\model")
MODEL_DIR.mkdir(parents=True, exist_ok=True)

model_path = MODEL_DIR / "pm10_lstm_forecasting.keras"

model.save(model_path)

print("모델 저장:", model_path)

## 17. 이번 장 정리

이번 장에서 배운 핵심은 다음과 같습니다.

```text
1. PM10 예측은 숫자를 맞히는 회귀 문제다.
2. LSTM 입력은 (samples, timesteps, features) 모양이다.
3. 출력층은 다음 PM10 값 하나를 내므로 Dense(1)을 사용한다.
4. 예측값은 inverse_transform으로 원래 PM10 단위로 되돌려야 한다.
5. 회귀 평가는 MAE와 RMSE로 해석할 수 있다.
```

## 과제

1. `LSTM(32)`를 `LSTM(16)` 또는 `LSTM(64)`로 바꾸면 어떤 변화가 예상되는지 적어보세요.
2. `WINDOW_SIZE`를 6 또는 72로 바꾸면 모델 입력 의미가 어떻게 달라지는지 설명해보세요.
3. MAE와 RMSE 중 RMSE가 큰 오차에 더 민감한 이유를 찾아 적어보세요.
4. 실제값/예측값 그래프에서 모델이 잘 못 따라가는 구간이 어떤 구간인지 관찰해보세요.